# 溯源追踪(W3C PROV-O)

## 概述

在医疗、法律、金融、科研等高风险领域,知识图谱的可信度取决于它能否回答**"这个事实从哪里来?"**。Semantica 的 `provenance` 模块为流经你管线的每一个实体、关系和分块提供审计级、与 W3C PROV-O 对齐的追踪。

在本 cookbook 中,你将学到如何:

- 追踪实体和关系,并附带**来源细节**(DOI、页码、原文引用、置信度)
- 遍历一个事实的完整**血缘**(文档 → 分块 → 实体 → 知识图谱)
- 审计某个实体背后的**修订历史**和**所有来源**
- 在**不删除**的前提下将事实**作废**(prov:Invalidation)—— 更正过程仍可被证明
- 使用链式 SHA-256 校验和验证**防篡改性**

**场景:**一个研究团队将两篇科学论文(带 DOI)的发现接入知识图谱。监管机构随后询问:*"哪篇论文、哪张图、哪一句原文支持了'鱼类生物量增长了 463%'这一说法?这个事实曾经被更正过吗?"*

In [ ]:
!pip install -q semantica

In [ ]:
import json
from semantica.provenance import (
    ProvenanceManager,
    compute_checksum,
    verify_checksum,
)

# In-memory storage for this demo; pass storage_path="provenance.db"
# (or a config with provenance.storage_path) for a persistent SQLite backend.
prov = ProvenanceManager()
print("ProvenanceManager ready (in-memory storage)")

## 步骤 1:追踪实体并附带审计级的来源细节

我们接入的每个事实都携带它的证据:**来源标识符**(这里是 DOI)、来源中的**位置**(某张图)、**原文引用**,以及抽取器的**置信度**。

In [ ]:
# Finding from paper #1
entry_biomass = prov.track_entity(
    entity_id="claim_biomass_increase",
    source="DOI:10.1371/journal.pone.0023601",
    confidence=0.92,
    source_location="Figure 2",
    source_quote="Total fish biomass increased by 463% ...",
)

# Supporting entity from paper #2
entry_reserve = prov.track_entity(
    entity_id="marine_reserve_1",
    source="DOI:10.1126/science.1088121",
    confidence=0.88,
    source_location="Table 1",
    source_quote="... no-take marine reserve at Cabo Pulmo ...",
)

print("Tracked:", entry_biomass.entity_id, "|", entry_reserve.entity_id)

## 步骤 2:追踪事实之间的关系

事实很少孤立存在。关于生物量增长的论断是*关于*海洋保护区的 —— 这种关系同样是一个一等公民的、受溯源追踪的对象。

`track_relationship()` 没有专门的主语/宾语字段,因此按照约定,我们在 `metadata` 中记录它连接的两个实体。

In [ ]:
rel = prov.track_relationship(
    relationship_id="rel_biomass_about_reserve",
    source="DOI:10.1371/journal.pone.0023601",
    metadata={
        "type": "measured_at",
        # No dedicated endpoint fields on track_relationship() yet -- record
        # which entities this relationship connects here by convention.
        "subject_entity_id": "claim_biomass_increase",
        "object_entity_id": "marine_reserve_1",
    },
)

print("Relationship tracked:", rel.entity_id, "|", rel.metadata["subject_entity_id"], "->", rel.metadata["object_entity_id"])

## 步骤 3:遍历血缘

`get_lineage` 会重建关于某个事实的所有已知信息;`trace_lineage` 返回有序的 `ProvenanceEntry` 记录链 —— 每个版本、每个活动、每个接触过它的智能体(agent)。

In [ ]:
lineage = prov.get_lineage("claim_biomass_increase")
print(json.dumps(lineage, indent=2, default=str)[:800])

print("\n--- ordered chain ---")
for e in prov.trace_lineage("claim_biomass_increase"):
    print(f"{e.entity_id} | seq#{e.sequence_id} | {e.activity_id}")

## 步骤 4:审计来源与修订历史

当监管机构问*"这个事实曾经被更正过吗?"*时,`revision_history` 会用完整的版本链来回答,而 `get_all_sources` 会列出曾经支持该实体的每一份源文档。

In [ ]:
revisions = prov.revision_history("claim_biomass_increase")
print(f"{len(revisions)} revision(s) on record")

for s in prov.get_all_sources("claim_biomass_increase"):
    print("source:", s)

## 步骤 5:作废 —— 更正而不删除

假设论文 #1 被部分撤回。审计轨迹**不能**悄悄删除该事实:`invalidate` 会归档作废前的状态,并追加一条新的 `prov:Invalidation` 条目,注明**谁**撤回了它以及**为什么**。

In [ ]:
invalidated = prov.invalidate(
    entity_id="claim_biomass_increase",
    agent_id="reviewer_dr_chen",
    reason="Partial retraction: Figure 2 statistics corrected by publisher (see erratum).",
)
print("Invalidated:", invalidated.entity_id, "| invalidated flag:", getattr(invalidated, "invalidated", True))

stats = prov.get_statistics()
print("\nStorage statistics:", json.dumps(stats, indent=2, default=str))

## 步骤 6:验证防篡改性

每个条目都带有一个确定性的 SHA-256 校验和,并与前一个条目链接。重新计算并比较,即可检测到溯源记录事后被破坏的情况。

In [ ]:
# entry_biomass was returned by track_entity in Step 1
ok = verify_checksum(entry_biomass)
print("Checksum verified:", ok)

print("Computed:", compute_checksum(entry_biomass)[:16], "...")
print("Stored:  ", entry_biomass.checksum[:16] if getattr(entry_biomass, 'checksum', None) else "(see entry fields)")
chain = prov.verify_chain()
print("Chain verification:", json.dumps(chain, default=str)[:200])


## 总结

| 需求 | 调用 |
|---|---|
| 记录事实的证据 | `prov.track_entity(entity_id, source, confidence=..., source_location=..., source_quote=...)` |
| 记录关系 | `prov.track_relationship(relationship_id, source, metadata=...)` |
| 事实的完整血缘 | `prov.get_lineage(entity_id)` / `prov.trace_lineage(entity_id)` |
| "它曾经被更正过吗?" | `prov.revision_history(entity_id)` |
| "哪些来源支持它?" | `prov.get_all_sources(entity_id)` |
| 撤回而不删除 | `prov.invalidate(entity_id, agent_id, reason=...)` |
| 防篡改检查 | `verify_checksum(entry)` |

### 接下来去哪

- **冲突检测与解决**(Notebook 17)—— 当两个来源不一致时会发生什么。
- **你的第一个知识图谱**(Notebook 08)—— 在抽取器中启用 `provenance=True`,让追踪在接入过程中自动发生。
- 模块文档字符串(`help(semantica.provenance)`)记录了与 `kg`、`split` 和 `conflicts` 追踪器的可选集成。